# M2_01 — Feature engineering 1D → feature sets (base/tech/macro/sent/full)

Objetivo (M2):
- Generar datasets 1D reproducibles por feature set.
- Aplicar política de missingness definida en el planning.

Inputs:
- `../data/_inputs/xbx_coindesk_ohlcv_1d_clean.csv`
- `../data/_inputs/external_1d__macro.csv` (opcional)
- `../data/_inputs/external_1d__sent.csv` (opcional)

Outputs:
- Datasets de entrenamiento (únicos CSV que quedan en `../data/`):
  - `../data/btc_1d_features__{base|tech|macro|sent|full}.csv`
- QA:
  - `../data/_qa/data_quality__btc_1d_features__{base|tech|macro|sent|full}.json`


In [1]:
import sys
import subprocess
import importlib
import pathlib
import json

for p in ["pandas", "numpy"]:
    try:
        importlib.import_module(p)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", p])

import numpy as np
import pandas as pd


In [2]:
base_dir = pathlib.Path(".").resolve()
data_dir = (base_dir.parent / "data").resolve()
inputs_dir = (data_dir / "_inputs").resolve()
qa_dir = (data_dir / "_qa").resolve()
inputs_dir.mkdir(parents=True, exist_ok=True)
qa_dir.mkdir(parents=True, exist_ok=True)

btc_1d_clean_path = inputs_dir / "xbx_coindesk_ohlcv_1d_clean.csv"
external_macro_path = inputs_dir / "external_1d__macro.csv"
external_sent_path = inputs_dir / "external_1d__sent.csv"

out_paths = {
    "base": data_dir / "btc_1d_features__base.csv",
    "tech": data_dir / "btc_1d_features__tech.csv",
    "macro": data_dir / "btc_1d_features__macro.csv",
    "sent": data_dir / "btc_1d_features__sent.csv",
    "full": data_dir / "btc_1d_features__full.csv",
}

qa_paths = {k: qa_dir / f"data_quality__btc_1d_features__{k}.json" for k in out_paths.keys()}

(btc_1d_clean_path.exists(), external_macro_path.exists(), external_sent_path.exists())

(True, True, True)

In [3]:
def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    rs = avg_gain / (avg_loss + 1e-12)
    return 100 - (100 / (1 + rs))


def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["log_close"] = np.log(df["close"].astype(float))
    df["log_return"] = df["log_close"].diff()
    df["log_volume"] = np.log1p(df["volume"].astype(float))
    return df


def add_tech_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for i in range(1, 8):
        df[f"log_ret_lag_{i}"] = df["log_return"].shift(i)
    df["volatility_7"] = df["log_return"].rolling(window=7).std(ddof=0)
    df["volatility_30"] = df["log_return"].rolling(window=30).std(ddof=0)
    df["rsi_14"] = compute_rsi(df["close"], period=14)
    for w in [8, 20, 100]:
        df[f"sma_{w}"] = df["close"].rolling(window=w).mean()
        df[f"ema_{w}"] = df["close"].ewm(span=w, adjust=False).mean()

    close = df["close"].astype(float)
    open_ = df["open"].astype(float)
    high = df["high"].astype(float)
    low = df["low"].astype(float)

    eps = 1e-12
    df["candle_range"] = (high - low) / (close.replace(0, np.nan))
    df["candle_body"] = (close - open_) / (open_.replace(0, np.nan))
    df["upper_wick"] = (high - np.maximum(open_, close)) / (close.replace(0, np.nan))
    df["lower_wick"] = (np.minimum(open_, close) - low) / (close.replace(0, np.nan))
    denom = (high - low).replace(0, np.nan)
    df["buying_pressure"] = (close - low) / denom
    df["buying_pressure"] = df["buying_pressure"].fillna(0.5)

    df["sin_day"] = np.sin(2 * np.pi * df["timestamp"].dt.dayofweek / 7)
    df["cos_day"] = np.cos(2 * np.pi * df["timestamp"].dt.dayofweek / 7)
    df["sin_month"] = np.sin(2 * np.pi * df["timestamp"].dt.month / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["timestamp"].dt.month / 12)
    return df


def apply_missingness_policy(df: pd.DataFrame, required_cols: list[str], allow_drop_cols: list[str]):
    warn_missing = 0.05
    drop_feature_missing = 0.20

    missing_frac = df.isna().mean().to_dict()
    dropped = []
    warned = [c for c, v in missing_frac.items() if v >= warn_missing]

    for c in allow_drop_cols:
        if missing_frac.get(c, 0.0) >= drop_feature_missing:
            dropped.append(c)
    df2 = df.drop(columns=dropped, errors="ignore")

    required_effective = [c for c in required_cols if c not in dropped]
    before_rows = int(len(df2))
    df3 = df2.dropna(subset=required_effective).reset_index(drop=True)
    after_rows = int(len(df3))

    report = {
        "warn_missing_threshold": warn_missing,
        "drop_feature_missing_threshold": drop_feature_missing,
        "warned_cols": warned,
        "dropped_cols": dropped,
        "required_cols": required_cols,
        "required_cols_effective": required_effective,
        "rows_before": before_rows,
        "rows_after": after_rows,
        "rows_dropped_fraction": float((before_rows - after_rows) / before_rows) if before_rows else 0.0,
    }
    return df3, report


In [4]:
df = pd.read_csv(btc_1d_clean_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)

df = add_base_features(df)
df = add_tech_features(df)

base_cols = ["timestamp", "open", "high", "low", "close", "volume", "log_close", "log_return", "log_volume"]
tech_extra_cols = [
    "rsi_14",
    "sma_8","ema_8","sma_20","ema_20","sma_100","ema_100",
    "volatility_7","volatility_30",
    "buying_pressure",
    "candle_range","candle_body","upper_wick","lower_wick",
    "log_ret_lag_1","log_ret_lag_2","log_ret_lag_3","log_ret_lag_4","log_ret_lag_5","log_ret_lag_6","log_ret_lag_7",
    "sin_day","cos_day","sin_month","cos_month",
]

macro_cols = []
sent_cols = []

if external_macro_path.exists():
    dm = pd.read_csv(external_macro_path)
    dm["timestamp"] = pd.to_datetime(dm["timestamp"], utc=True)
    dm = dm.sort_values("timestamp")
    macro_cols = [c for c in dm.columns if c != "timestamp"]
    df = df.merge(dm, on="timestamp", how="left")

if external_sent_path.exists():
    ds = pd.read_csv(external_sent_path)
    ds["timestamp"] = pd.to_datetime(ds["timestamp"], utc=True)
    ds = ds.sort_values("timestamp")
    sent_cols = [c for c in ds.columns if c != "timestamp"]
    df = df.merge(ds, on="timestamp", how="left")

df.shape, len(macro_cols), len(sent_cols)

((4151, 50), 12, 2)

In [5]:
def export_feature_set(name: str, cols: list[str], required_cols: list[str], allow_drop_cols: list[str]):
    df_set = df[[c for c in cols if c in df.columns]].copy()
    df_set, policy_report = apply_missingness_policy(df_set, required_cols=required_cols, allow_drop_cols=allow_drop_cols)
    df_set = df_set.reset_index(drop=True)

    out_df = df_set.copy()
    out_df["timestamp"] = pd.to_datetime(out_df["timestamp"], utc=True).dt.strftime("%Y-%m-%d %H:%M:%S+00:00")
    out_df.to_csv(out_paths[name], index=False)

    qa = {
        "feature_set": name,
        "rows": int(len(out_df)),
        "cols": int(out_df.shape[1]),
        "start": (out_df["timestamp"].iloc[0] if len(out_df) else None),
        "end": (out_df["timestamp"].iloc[-1] if len(out_df) else None),
        "missing_fraction": {k: float(v) for k, v in df_set.isna().mean().to_dict().items()},
        "missingness_policy": policy_report,
    }
    qa_paths[name].write_text(json.dumps(qa, ensure_ascii=False, indent=2), encoding="utf-8")
    print(name, 'rows', qa['rows'], 'cols', qa['cols'], 'start', qa['start'], 'end', qa['end'])


export_feature_set(
    name="base",
    cols=base_cols,
    required_cols=["timestamp", "open", "high", "low", "close", "volume", "log_close", "log_return", "log_volume"],
    allow_drop_cols=[],
)

export_feature_set(
    name="tech",
    cols=base_cols + tech_extra_cols,
    required_cols=["timestamp", "open", "high", "low", "close", "volume", "log_return", "rsi_14", "sma_100", "ema_100", "volatility_30"],
    allow_drop_cols=[],
)

if macro_cols:
    export_feature_set(
        name="macro",
        cols=base_cols + macro_cols,
        required_cols=["timestamp", "open", "high", "low", "close", "volume"] + [c for c in macro_cols],
        allow_drop_cols=list(macro_cols),
    )

if sent_cols:
    export_feature_set(
        name="sent",
        cols=base_cols + sent_cols,
        required_cols=["timestamp", "open", "high", "low", "close", "volume", "fgi"],
        allow_drop_cols=[c for c in sent_cols if c not in ["fgi", "fgi_norm"]],
    )

if macro_cols or sent_cols:
    full_cols = base_cols + tech_extra_cols + macro_cols + sent_cols
    required_full = ["timestamp", "open", "high", "low", "close", "volume", "log_return"]
    if macro_cols:
        required_full += macro_cols
    if sent_cols:
        required_full += ["fgi"]
    export_feature_set(
        name="full",
        cols=full_cols,
        required_cols=required_full,
        allow_drop_cols=list(macro_cols) + [c for c in sent_cols if c not in ["fgi", "fgi_norm"]],
    )


base rows 4150 cols 9 start 2014-11-04 00:00:00+00:00 end 2026-03-15 00:00:00+00:00
tech rows 4052 cols 34 start 2015-02-10 00:00:00+00:00 end 2026-03-15 00:00:00+00:00
macro rows 4144 cols 21 start 2014-11-10 00:00:00+00:00 end 2026-03-15 00:00:00+00:00
sent rows 2961 cols 11 start 2018-02-01 00:00:00+00:00 end 2026-03-15 00:00:00+00:00


full rows 2961 cols 48 start 2018-02-01 00:00:00+00:00 end 2026-03-15 00:00:00+00:00
